In [1]:
import logging
import time
from pathlib import Path
from collections import deque
from typing import Dict, Any
import numpy as np

import gymnasium as gym
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecNormalize
from stable_baselines3.common.callbacks import BaseCallback, CallbackList
import src.gymnasium_envs.nn_optimization_env

PROJECT_ROOT = Path().resolve().parent.parent

log_dir = PROJECT_ROOT / "logs"
model_dir = PROJECT_ROOT / "models"

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Using device: {device}")
if torch.cuda.is_available():
    logger.info(f"GPU: {torch.cuda.get_device_name(0)}")
    logger.info(f"CUDA Version: {torch.version.cuda}")

2026-06-09 08:43:03 [INFO] Using device: cuda
2026-06-09 08:43:03 [INFO] GPU: NVIDIA GeForce GTX 1650
2026-06-09 08:43:03 [INFO] CUDA Version: 11.8


In [3]:
class StatusCallback(BaseCallback):
    def __init__(self, window_size=1000):
        super().__init__()
        self.window = deque(maxlen=window_size)

    def _on_step(self):
        infos = self.locals["infos"]
        for info in infos:
            status = info.get("status")
            if status == "converged":
                self.window.append(1)
            elif status == "diverged":
                self.window.append(-1)

        return True

In [13]:
def train_model(config: Dict[str, Any], log_dir: str = "logs", model_dir: str = "models", add_time_penalty : bool = False, learn_betas : bool = False, name_prefix = ""):
    task_name = name_prefix + "nn"

    logger.info(f"Starting training for {task_name} optimization task. Timesteps: {config['timesteps']}, Envs: {config['n_envs']}")

    start_time    = time.time()
    model_path    = Path(f"{model_dir}/{task_name}")
    stats_path    = Path(f"{model_dir}/{task_name}_vec_normalize.pkl")

    model_path.parent.mkdir(parents=True, exist_ok=True)
    vec_env = None
    try:
        vec_env = make_vec_env(
            "nn_optimization_env/NeuralNetworkOptimization-v1",
            n_envs=config["n_envs"],
            env_kwargs={"max_iterations" : config["max_iterations"],
                        "add_time_penalty" : add_time_penalty,
                        "learn_betas" : learn_betas,
                        "dataset_name" : "MNIST"}
        )

        vec_env = VecNormalize(vec_env, norm_obs=False, norm_reward=True)

        model = PPO(
            "MultiInputPolicy",
            vec_env,
            verbose=1,
            tensorboard_log=f"{log_dir}/{task_name}d/",
            n_steps=config["n_steps"],
            n_epochs=config["n_epochs"],
            batch_size=config["batch_size"],
            learning_rate=config["learning_rate"],
            gamma=config["gamma"],
            clip_range=config["clip_range"],
            ent_coef=config["ent_coef"],
            policy_kwargs=config["policy_kwargs"]
        )

        model.learn(
            total_timesteps=config["timesteps"],
            tb_log_name=f"{task_name}",
            callback=StatusCallback()
        )

        # Сохраняем финальную модель отдельно
        model.save(str(model_path))
        vec_env.save(str(stats_path))

        elapsed_time = (time.time() - start_time) / 60
        logger.info(f"Training for {task_name} completed. Duration: {elapsed_time:.2f} min.")
        logger.info(f"Final model saved to: {model_path}")

    except Exception as e:
        logger.error(f"Error during training for {task_name}: {str(e)}", exc_info=True)
    finally:
        if vec_env is not None:
            vec_env.close()

In [18]:
config_base = {
    "max_iterations": 3000,
    "timesteps"     : 1_000_000, 
    "n_envs"        : 32,
    "n_steps"       : 2048,
    "n_epochs"      : 10, 
    "batch_size"    : 256, 
    "learning_rate" : 3e-4,
    "gamma"         : 0.99,
    "clip_range"    : 0.2,
    "ent_coef"      : 0.01,
    "policy_kwargs" : {
        "net_arch": dict(pi=[64, 64], vf=[64, 64])
    }
}

In [19]:
config = config_base

train_model(config, log_dir, model_dir)

2026-06-09 14:36:01 [INFO] Starting training for nn optimization task. Timesteps: 1000000, Envs: 32


Using cuda device
Logging to C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\logs/nnd/nn_3
------------------------------
| time/              |       |
|    fps             | 29    |
|    iterations      | 1     |
|    time_elapsed    | 2247  |
|    total_timesteps | 65536 |
------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 3e+03        |
|    ep_rew_mean          | -6.22        |
| time/                   |              |
|    fps                  | 31           |
|    iterations           | 2            |
|    time_elapsed         | 4189         |
|    total_timesteps      | 131072       |
| train/                  |              |
|    approx_kl            | 0.0031230275 |
|    clip_fraction        | 0.0198       |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.41        |
|    explained_variance   | -0.61        |
|    learning_rate 

2026-06-09 17:49:13 [INFO] Training for nn completed. Duration: 193.19 min.
2026-06-09 17:49:13 [INFO] Final model saved to: C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\models\nn


In [20]:
config = config_base

config["max_iterations"] = 1000
config["timesteps"] = 3000000

train_model(config, log_dir, model_dir, name_prefix="less_iters_")

2026-06-09 17:49:13 [INFO] Starting training for less_iters_nn optimization task. Timesteps: 3000000, Envs: 32


Using cuda device
Logging to C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\logs/less_iters_nnd/less_iters_nn_1
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1e+03    |
|    ep_rew_mean     | -6.04    |
| time/              |          |
|    fps             | 135      |
|    iterations      | 1        |
|    time_elapsed    | 482      |
|    total_timesteps | 65536    |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 1e+03       |
|    ep_rew_mean          | -5.73       |
| time/                   |             |
|    fps                  | 133         |
|    iterations           | 2           |
|    time_elapsed         | 983         |
|    total_timesteps      | 131072      |
| train/                  |             |
|    approx_kl            | 0.005360475 |
|    clip_fraction        | 0.0495      |
|    clip_range

2026-06-10 00:25:52 [INFO] Training for less_iters_nn completed. Duration: 396.64 min.
2026-06-10 00:25:52 [INFO] Final model saved to: C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\models\less_iters_nn


In [21]:
config = config_base

train_model(config, log_dir, model_dir, learn_betas=True, name_prefix="learn_betas_")

2026-06-10 00:25:52 [INFO] Starting training for learn_betas_nn optimization task. Timesteps: 3000000, Envs: 32


Using cuda device
Logging to C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\logs/learn_betas_nnd/learn_betas_nn_1
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1e+03    |
|    ep_rew_mean     | -5.49    |
| time/              |          |
|    fps             | 173      |
|    iterations      | 1        |
|    time_elapsed    | 378      |
|    total_timesteps | 65536    |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 1e+03       |
|    ep_rew_mean          | -5.4        |
| time/                   |             |
|    fps                  | 168         |
|    iterations           | 2           |
|    time_elapsed         | 777         |
|    total_timesteps      | 131072      |
| train/                  |             |
|    approx_kl            | 0.005281416 |
|    clip_fraction        | 0.0511      |
|    clip_ran

2026-06-10 05:28:04 [INFO] Training for learn_betas_nn completed. Duration: 302.20 min.
2026-06-10 05:28:04 [INFO] Final model saved to: C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\models\learn_betas_nn
